In [0]:
%fs head /Volumes/workspace/proyecto_bardero21/pruebas/datoss/FACT_VyENTAS.csv

[Truncated to first 65536 bytes]
﻿ID_Venta;ID_Sede;ID_Producto;ID_Categoria;ID_Empleado;Fecha;Cantidad;Ingreso_Total;Costo_Insumos;Gasto_Personal_Asignado;Gasto_Local_Asignado;Costo_Total
439;0;4;0;1;sábado, 6 de abril de 2024;4;112;48;26.5;15;89.5
440;0;1;0;1;sábado, 6 de abril de 2024;6;150;60;26.5;15;101.5
663;0;21;4;1;martes, 5 de marzo de 2024;4;60;20;26.5;15;61.5
664;0;30;6;1;martes, 5 de marzo de 2024;6;72;24;26.5;15;65.5
665;0;32;6;1;martes, 5 de marzo de 2024;4;40;12;26.5;15;53.5
666;0;39;8;1;martes, 5 de marzo de 2024;5;150;60;26.5;15;101.5
743;0;0;0;1;viernes, 12 de abril de 2024;6;210;84;26.5;15;125.5
768;0;17;3;1;sábado, 17 de febrero de 2024;5;175;90;26.5;15;131.5
769;0;4;0;1;sábado, 17 de febrero de 2024;5;140;60;26.5;15;101.5
770;0;5;1;1;sábado, 17 de febrero de 2024;6;150;108;26.5;15;149.5
990;0;0;0;1;jueves, 9 de mayo de 2024;4;140;56;26.5;15;97.5
991;0;4;0;1;jueves, 9 de mayo de 2024;4;112;48;26.5;15;89.5
992;0;8;1;1;jueves, 9 de mayo de 2024;5;225;110;26.5;15;151.5


In [0]:
dfVentas = spark.read.csv("/Volumes/workspace/proyecto_bardero21/pruebas/datoss/FACT_VENTAS.csv", header= True, inferSchema= True, sep = ";")

In [0]:
display(dfVentas)

ID_Venta,ID_Sede,ID_Producto,ID_Categoria,ID_Empleado,Fecha,Cantidad,Ingreso_Total,Costo_Insumos,Gasto_Personal_Asignado,Gasto_Local_Asignado,Costo_Total
439,0,4,0,1,"sábado, 6 de abril de 2024",4,112,48,26.5,15.0,89.5
440,0,1,0,1,"sábado, 6 de abril de 2024",6,150,60,26.5,15.0,101.5
663,0,21,4,1,"martes, 5 de marzo de 2024",4,60,20,26.5,15.0,61.5
664,0,30,6,1,"martes, 5 de marzo de 2024",6,72,24,26.5,15.0,65.5
665,0,32,6,1,"martes, 5 de marzo de 2024",4,40,12,26.5,15.0,53.5
666,0,39,8,1,"martes, 5 de marzo de 2024",5,150,60,26.5,15.0,101.5
743,0,0,0,1,"viernes, 12 de abril de 2024",6,210,84,26.5,15.0,125.5
768,0,17,3,1,"sábado, 17 de febrero de 2024",5,175,90,26.5,15.0,131.5
769,0,4,0,1,"sábado, 17 de febrero de 2024",5,140,60,26.5,15.0,101.5
770,0,5,1,1,"sábado, 17 de febrero de 2024",6,150,108,26.5,15.0,149.5


In [0]:
from pyspark.sql import functions as F

print("Total filas originales:", dfVentas.count())
dfVentas.printSchema()

# Revisar nulos por columna
dfVentas.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in dfVentas.columns
]).show(truncate=False)

Total filas originales: 25916
root
 |-- ID_Venta: integer (nullable = true)
 |-- ID_Sede: integer (nullable = true)
 |-- ID_Producto: integer (nullable = true)
 |-- ID_Categoria: integer (nullable = true)
 |-- ID_Empleado: integer (nullable = true)
 |-- Fecha: string (nullable = true)
 |-- Cantidad: integer (nullable = true)
 |-- Ingreso_Total: integer (nullable = true)
 |-- Costo_Insumos: integer (nullable = true)
 |-- Gasto_Personal_Asignado: double (nullable = true)
 |-- Gasto_Local_Asignado: double (nullable = true)
 |-- Costo_Total: double (nullable = true)

+--------+-------+-----------+------------+-----------+-----+--------+-------------+-------------+-----------------------+--------------------+-----------+
|ID_Venta|ID_Sede|ID_Producto|ID_Categoria|ID_Empleado|Fecha|Cantidad|Ingreso_Total|Costo_Insumos|Gasto_Personal_Asignado|Gasto_Local_Asignado|Costo_Total|
+--------+-------+-----------+------------+-----------+-----+--------+-------------+-------------+--------------------

In [0]:
df_clean = (
    dfVentas
    .withColumn("Sede_Label", F.concat(F.lit("Sede "), F.col("ID_Sede").cast("string")))
    .withColumn("Ingreso_Num", F.regexp_replace("Ingreso_Total", ",", ".").cast("double"))
    .withColumn("Costo_Num", F.regexp_replace("Costo_Total", ",", ".").cast("double"))
)

In [0]:
df_sede = (
    df_clean.groupBy("Sede_Label")
    .agg(
        F.count("*").alias("Total_Filas"),
        F.round(F.sum("Ingreso_Num"), 2).alias("Suma_Ingresos"),
        F.round(F.sum("Costo_Num"), 2).alias("Suma_Costos"),
    )
    .orderBy("Sede_Label")
)

display(df_sede)

Sede_Label,Total_Filas,Suma_Ingresos,Suma_Costos
Sede 0,8744,936693.0,789269.42
Sede 1,8564,909955.0,749749.33
Sede 2,8608,921709.0,745060.44


In [0]:
display(df_clean)

ID_Venta,ID_Sede,ID_Producto,ID_Categoria,ID_Empleado,Fecha,Cantidad,Ingreso_Total,Costo_Insumos,Gasto_Personal_Asignado,Gasto_Local_Asignado,Costo_Total,Sede_Label,Ingreso_Num,Costo_Num
439,0,4,0,1,"sábado, 6 de abril de 2024",4,112,48,26.5,15.0,89.5,Sede 0,112.0,89.5
440,0,1,0,1,"sábado, 6 de abril de 2024",6,150,60,26.5,15.0,101.5,Sede 0,150.0,101.5
663,0,21,4,1,"martes, 5 de marzo de 2024",4,60,20,26.5,15.0,61.5,Sede 0,60.0,61.5
664,0,30,6,1,"martes, 5 de marzo de 2024",6,72,24,26.5,15.0,65.5,Sede 0,72.0,65.5
665,0,32,6,1,"martes, 5 de marzo de 2024",4,40,12,26.5,15.0,53.5,Sede 0,40.0,53.5
666,0,39,8,1,"martes, 5 de marzo de 2024",5,150,60,26.5,15.0,101.5,Sede 0,150.0,101.5
743,0,0,0,1,"viernes, 12 de abril de 2024",6,210,84,26.5,15.0,125.5,Sede 0,210.0,125.5
768,0,17,3,1,"sábado, 17 de febrero de 2024",5,175,90,26.5,15.0,131.5,Sede 0,175.0,131.5
769,0,4,0,1,"sábado, 17 de febrero de 2024",5,140,60,26.5,15.0,101.5,Sede 0,140.0,101.5
770,0,5,1,1,"sábado, 17 de febrero de 2024",6,150,108,26.5,15.0,149.5,Sede 0,150.0,149.5
